# Milestone 2: Data Quality dan Profiling 

Lanjutan dari `01_data_profiling.ipynb` untuk **ID_REG_Parsed — Indonesian Regulation Parsed Dataset**. Proses memakai Polars Lazy API agar aman untuk 3,6 juta baris data.

In [2]:
from pathlib import Path

import duckdb
import polars as pl

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'data').exists() else CWD.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw' / 'DataSet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'id_reg_clean.parquet'
PARQUET_FILES = sorted(RAW_DIR.glob('*.parquet'))
if not PARQUET_FILES:
    raise FileNotFoundError(f'Tidak ada file Parquet di {RAW_DIR}')

# Daftar file eksplisit lebih stabil daripada glob pada Polars/Windows untuk banyak file.
lf_raw = pl.scan_parquet(PARQUET_FILES, low_memory=True)
COLUMNS = lf_raw.collect_schema().names()
TOTAL_ROWS = lf_raw.select(pl.len()).collect().item()
print(f'Data awal: {TOTAL_ROWS:,} baris x {len(COLUMNS)} kolom')
lf_raw.head()

Data awal: 3,630,932 baris x 7 kolom


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom dari sampel 100.000 baris agar penggunaan memori tetap terjaga. Cleaning tetap memproses seluruh data.

In [3]:
PROFILE_SAMPLE_ROWS = 100_000
df_sample = lf_raw.head(PROFILE_SAMPLE_ROWS).collect(engine='streaming')
connection = duckdb.connect()
connection.register('dataset', df_sample.to_arrow())

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""Regulation Name""","""VARCHAR""","""'LembaramNegara Republik Indon…","""• BUPATI SITUBONDO""",979,null,null,null,null,null,100000,0.40
"""Regulation Number""","""VARCHAR""","""!""","""��""",759,null,null,null,null,null,100000,0.05
"""Year""","""VARCHAR""","""1945""","""2020""",72,null,null,null,null,null,100000,0.00
"""About""","""VARCHAR""","""!SALi NAN\""","""·,\H/\N AT/\S PERATURAN BUPATI…",4264,null,null,null,null,null,100000,9.43
"""Chapter""","""VARCHAR""","""BAB I - II""","""bab i - ni""",8508,null,null,null,null,null,100000,16.24
"""Article""","""VARCHAR""","""PASAL 1""","""pasal 98""",1049,null,null,null,null,null,100000,0.00
"""Content""","""VARCHAR""","""! (1) Pengurangan sampah seba…","""��is Retribusi Pelayanan Park…",69254,null,null,null,null,null,100000,0.02


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [4]:
null_counts = lf_raw.select(pl.len().alias('_jumlah_baris'), *[pl.col(c).null_count().alias(c) for c in COLUMNS]).collect(engine='streaming')
null_summary = (
    null_counts.drop('_jumlah_baris')
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / TOTAL_ROWS * 100).round(4).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""Chapter""",639692,17.6178
"""About""",166460,4.5845
"""Regulation Name""",12376,0.3408
"""Content""",4421,0.1218
"""Regulation Number""",1754,0.0483
"""Year""",580,0.016
"""Article""",17,0.0005


## 3. Penanganan Missing Values

Semua kolom teks dirapikan (whitespace dan string kosong). `Chapter` kosong diberi label `TANPA BAB`, `About` kosong diberi `TIDAK DIKETAHUI`; data tanpa metadata inti atau isi regulasi disaring.

In [5]:
TEXT_COLUMNS = ['Regulation Name', 'Regulation Number', 'Year', 'About', 'Chapter', 'Article', 'Content']
REQUIRED_COLUMNS = ['Regulation Name', 'Regulation Number', 'Year', 'Content']

def clean_text(column):
    return (pl.col(column).cast(pl.String).str.replace_all(r'\s+', ' ')
            .str.strip_chars().replace('', None).alias(column))

lf_missing_handled = (
    lf_raw.with_columns([clean_text(c) for c in TEXT_COLUMNS])
    .with_columns([pl.col('Chapter').fill_null('TANPA BAB'),
                   pl.col('About').fill_null('TIDAK DIKETAHUI'),
                   pl.col('Year').cast(pl.Int32, strict=False).alias('Year')])
    .filter(pl.all_horizontal(*[pl.col(c).is_not_null() for c in REQUIRED_COLUMNS]))
    .filter(pl.col('Year').is_between(1700, 2100))
)
print('Cleaning missing values disiapkan secara lazy; hasil dihitung pada sel deduplikasi.')

Cleaning missing values disiapkan secara lazy; hasil dihitung pada sel deduplikasi.


## 4. Deteksi dan Penanganan Outlier

Dataset tidak memiliki variabel numerik kontinu. Metode IQR diterapkan pada panjang `Content`; teks panjang dilaporkan tetapi tidak dihapus karena bisa merupakan lampiran regulasi yang valid.

In [6]:
# Tidak ada kolom numerik kontinu; outlier diukur dari panjang teks Content.
# Quantile seluruh data dapat melebihi memori, sehingga gunakan sampel deterministik.
from pathlib import Path
import polars as pl

OUTLIER_SAMPLE_ROWS = 25_000
OUTLIER_SAMPLE_FILES = 3
# Tetap dapat dijalankan setelah kernel restart, meskipun sel inisialisasi belum dijalankan ulang.
if 'PARQUET_FILES' not in globals():
    CWD = Path.cwd().resolve()
    PROJECT_ROOT = CWD if (CWD / 'data').exists() else CWD.parent
    PARQUET_FILES = sorted((PROJECT_ROOT / 'data' / 'raw' / 'DataSet').glob('*.parquet'))
if not PARQUET_FILES:
    raise FileNotFoundError('File Parquet tidak ditemukan di data/raw/DataSet')
if 'lf_missing_handled' not in globals():
    TEXT_COLUMNS = ['Regulation Name', 'Regulation Number', 'Year', 'About', 'Chapter', 'Article', 'Content']
    REQUIRED_COLUMNS = ['Regulation Name', 'Regulation Number', 'Year', 'Content']
    def clean_text(column):
        return (pl.col(column).cast(pl.String).str.replace_all(r'\s+', ' ')
                .str.strip_chars().replace('', None).alias(column))
    lf_missing_handled = (
        pl.scan_parquet(PARQUET_FILES, low_memory=True)
        .with_columns([clean_text(c) for c in TEXT_COLUMNS])
        .with_columns([pl.col('Chapter').fill_null('TANPA BAB'),
                       pl.col('About').fill_null('TIDAK DIKETAHUI'),
                       pl.col('Year').cast(pl.Int32, strict=False).alias('Year')])
        .filter(pl.all_horizontal(*[pl.col(c).is_not_null() for c in REQUIRED_COLUMNS]))
        .filter(pl.col('Year').is_between(1700, 2100))
    )
lf_outlier_sample = (
    pl.scan_parquet(PARQUET_FILES[:OUTLIER_SAMPLE_FILES], low_memory=True)
    .select(pl.col('Content').cast(pl.String).str.replace_all(r'\s+', ' ').str.strip_chars().alias('Content'))
    .filter(pl.col('Content').is_not_null() & (pl.col('Content') != ''))
    .head(OUTLIER_SAMPLE_ROWS)
    .with_columns(pl.col('Content').str.len_chars().alias('content_length'))
)
length_stats = lf_outlier_sample.select(
    pl.col('content_length').quantile(0.25).alias('q1'),
    pl.col('content_length').quantile(0.75).alias('q3'),
    pl.len().alias('ukuran_sampel'),
).collect(engine='streaming')
q1, q3, ukuran_sampel = length_stats.row(0)
iqr = q3 - q1
batas_outlier = q3 + 1.5 * iqr
jumlah_outlier_sampel = (
    lf_outlier_sample.filter(pl.col('content_length') > batas_outlier)
    .select(pl.len()).collect(engine='streaming').item()
)
print(f'Sampel: {ukuran_sampel:,} teks dari {OUTLIER_SAMPLE_FILES} file pertama')
print(f'Q1={q1:,.0f}; Q3={q3:,.0f}; IQR={iqr:,.0f}; batas outlier={batas_outlier:,.0f}')
print(f'Potensi outlier pada sampel: {jumlah_outlier_sampel:,}; tidak dihapus karena dapat berupa lampiran regulasi.')
lf_outlier_handled = lf_missing_handled

Sampel: 25,000 teks dari 3 file pertama
Q1=73; Q3=721; IQR=648; batas outlier=1,693
Potensi outlier pada sampel: 2,247; tidak dihapus karena dapat berupa lampiran regulasi.


## 5. Deduplikasi

Duplikasi diperiksa memakai metadata unit regulasi dan `Content`, sehingga pasal berbeda dengan metadata serupa tidak ikut terhapus.

In [7]:
DEDUP_KEY = ['Regulation Name', 'Regulation Number', 'Year', 'Chapter', 'Article', 'Content']
rows_after_rules = lf_outlier_handled.select(pl.len()).collect(engine='streaming').item()
lf_dedup = lf_outlier_handled.unique(subset=DEDUP_KEY, keep='first')
rows_clean = lf_dedup.select(pl.len()).collect(engine='streaming').item()
print(f'Baris setelah aturan missing/tahun: {rows_after_rules:,}')
print(f'Duplikat identik dihapus: {rows_after_rules - rows_clean:,}')
print(f'Baris setelah deduplikasi: {rows_clean:,}')

Baris setelah aturan missing/tahun: 3,611,845
Duplikat identik dihapus: 29,271
Baris setelah deduplikasi: 3,582,574


## Catatan Milestone 2

- Masalah dari profiling: null pada metadata/isi, tahun tidak valid, variasi whitespace, dan potensi unit identik.
- Missing value: teks dinormalisasi; `Chapter`/`About` diberi label; baris tanpa identitas utama atau isi disaring.
- Outlier: panjang teks dilaporkan, tidak di-clipping agar isi regulasi tetap utuh.
- Deduplikasi: hanya unit dengan metadata dan isi identik yang dihapus.